### Data Cleaning

#### Purpose

Clean the selected Olist tables, then plan the joins for an analytical base table (ABT) with one row per order. The join plan will guide the second script in pipelines/.

#### Scope

- Check data types, missing values, and duplicate keys at each table's expected grain.Preserve leading zeros in ZIP code prefixes and parse order timestamps.
- Review missing approval and delivery timestamps in delivered orders and document how they affect eligibility for training and target construction.
- Decide how to handle missing product categories and multiple geolocation records per ZIP code prefix before joining the tables.
- Document what each column represents and how it will be used.
- Document the cleaning rules and their effects on row counts and key uniqueness.
- Plan the join keys, expected cardinality, and any aggregation needed to preserve one row per order.

In [1]:
import pandas as pd

#### customers

The customers table has 99,441 rows and no missing values across its four columns.

- customer_id: the table key and will be used to join customer information to orders on orders.customer_id.
- customer_zip_code_prefix: will link to the geolocation table to obtain destination coordinates for estimating seller-to-customer distances. Keep it as text to preserve leading zeros.
- customer_city and customer_state: candidate categorical features.

The customers table needs no further cleaning at this stage.

In [13]:
df_customers = pd.read_csv(
    "../data/selected/customers_dataset.csv",
    dtype={"customer_zip_code_prefix": "str"}
)
df_customers.info()
display(df_customers.head())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_zip_code_prefix  99441 non-null  str  
 2   customer_city             99441 non-null  str  
 3   customer_state            99441 non-null  str  
dtypes: str(4)
memory usage: 3.0 MB


,customer_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,13056,campinas,SP


#### Geolocation

The geolocation table has 1,000,163 rows and no missing values across its three columns. It contains 19,015 distinct ZIP code prefixes, and 17,972 of them appear in more than one row.

- geolocation_zip_code_prefix: joins to the ZIP code prefix in customers and sellers. Read all three prefix columns as text to preserve leading zeros. A prefix covers an area and can have several recorded locations.
- geolocation_lat and geolocation_lng: numeric coordinates that can be used as features or combined to estimate straight-line distances between sellers and customers.

The table contain repetead zip code prefixes because in Brazils zip code policy a zip code is formated like that XXXXX-YYY, this table only offers XXXXX without the last part, hence why it has prefix in it's name.

To deal with this repetition problem I first droped exact duplicates using the prefix, latitude, and longitude together. This removes 280,009 rows and leaves 720,154 distinct prefix and coordinate combinations. Within each prefix, each distinct location has equal weight in the next step.

Then I did a group by prefix and take the median of latitude and longitude separately. The median is less sensitive to unusually distant points and gives one representative coordinate pair per prefix. The resulting lookup has 19,015 rows. A unique prefix key prevents the geolocation joins from multiplying customer or seller rows.

Distances based on these coordinates are approximate. For prefix 24220, 90% of the distinct points are within about 795 meters of the median coordinate. For 639 prefixes, the radius needed to cover 90% of their distinct points exceeds 10 km.

Left join the lookup to customers and sellers with validate="many_to_one" to enforce uniqueness on the lookup side. Bring customer coordinates into the orders join and seller coordinates into the order-items join, where seller-to-customer distances can be calculated before aggregating to one row per order. The 4,099 geolocation prefixes absent from both tables can stay in the lookup because they will have no matches. There are also 157 distinct customer prefixes and 7 distinct seller prefixes with no geolocation match. The left joins preserve those records and leave their coordinates missing.

In [34]:
df_geol = pd.read_csv(
    "../data/selected/geolocations_dataset.csv",
    dtype={"geolocation_zip_code_prefix": "str"}
)
df_geol.info()
display(df_geol.head())

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 3 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  str    
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
dtypes: float64(2), str(1)
memory usage: 22.9 MB


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,01037,-23.545621,-46.639292
1,01046,-23.546081,-46.644820
2,01046,-23.546129,-46.642951
3,01041,-23.544392,-46.639499
4,01035,-23.541578,-46.641607


In [35]:
df_geol = df_geol.drop_duplicates(
    subset=[
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ]
)
df_geol.info()

<class 'pandas.DataFrame'>
Index: 720154 entries, 0 to 1000161
Data columns (total 3 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   geolocation_zip_code_prefix  720154 non-null  str    
 1   geolocation_lat              720154 non-null  float64
 2   geolocation_lng              720154 non-null  float64
dtypes: float64(2), str(1)
memory usage: 22.0 MB


In [36]:
df_geol = df_geol.groupby(
    by="geolocation_zip_code_prefix",
    as_index=False,

)[["geolocation_lat", "geolocation_lng"]].median()

df_geol.info()

<class 'pandas.DataFrame'>
RangeIndex: 19015 entries, 0 to 19014
Data columns (total 3 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   geolocation_zip_code_prefix  19015 non-null  str    
 1   geolocation_lat              19015 non-null  float64
 2   geolocation_lng              19015 non-null  float64
dtypes: float64(2), str(1)
memory usage: 445.8 KB


#### Order Items

The order items table has 112,650 rows and no missing values across its six columns. Each row represents one item, identified by the combination of order_id and order_item_id.

- order_id: links items to orders and is the grouping key for order-level features.
- order_item_id: identifies an item within its order. Count item rows to get the number of items per order.
- product_id: links each item to its product attributes.
- seller_id: links each item to its seller and the seller's coordinates.
- price: item price. Sum it by order to get the total merchandise value.
- freight_value: freight allocated to the item. Sum it by order to get the total freight value.

Group the item rows by order_id to calculate these candidate features for the ABT.

- item_count counts all item rows, including repeated units of the same product.
- distinct_product_count counts unique product_id values to describe the variety of products in the order.
- distinct_seller_count counts unique seller_id values to show how many sellers are involved in fulfilling the order.
- price_sum adds the item prices to get the total merchandise value.
- freight_sum adds the item freight amounts to get the total freight value.

These features use the order items table alone. For features that need product attributes or seller coordinates, join that information to the item rows before grouping. Join the aggregated results to the ABT with one row per order.

In [37]:
df_items = pd.read_csv("../data/selected/order_items_dataset.csv")
df_items.info()
display(df_items.head())

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_id       112650 non-null  str    
 1   order_item_id  112650 non-null  int64  
 2   product_id     112650 non-null  str    
 3   seller_id      112650 non-null  str    
 4   price          112650 non-null  float64
 5   freight_value  112650 non-null  float64
dtypes: float64(2), int64(1), str(3)
memory usage: 5.2 MB


,order_id,order_item_id,product_id,seller_id,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.90,18.14


#### Orders

The selected orders table has 99,441 rows and one row per order_id. It is the starting point for the ABT.

- order_id: the table key and the join key for the order-level summaries from items.
- customer_id: joins the order to its customer and destination information.
- order_status: used to select delivered orders. Drop it after filtering because it is constant in that subset.
- order_purchase_timestamp: records when the purchase was placed.
- order_approved_at: defines when the model scores the order. Use it for the chronological split and to calculate the calendar days from approval to the estimated delivery date.
- order_delivered_customer_date: the actual delivery timestamp, used to build the label and check when the outcome was available for training.
- order_estimated_delivery_date: the date to compare against actual delivery. Use the estimate available at approval to calculate the days remaining.

Filtering to delivered orders leaves 96,478 rows. Of those, 14 are missing the approval timestamp and 8 are missing the actual delivery timestamp. The two counts cover separate rows. Dropping those 22 rows leaves 96,456 complete orders. The model trained on this population estimates late-delivery risk among orders that eventually get delivered.

Compare the actual and estimated delivery dates at calendar-day granularity. An order is late when its actual delivery date is after the estimated date. Delivery on the estimated date counts as on time.

Parse the approval, actual delivery, and estimated delivery timestamps before checking their order. Exclude orders whose actual delivery timestamp precedes approval or whose estimated delivery date is before the approval date. The first check uses full timestamps, and the second uses calendar days so orders approved on their estimated delivery date remain eligible.

Among the 96,456 complete delivered orders, 61 were recorded as delivered before approval and 4 had an estimate before the approval date. Three met both conditions, so these checks exclude 62 distinct orders. The cleaned table has 96,394 delivered orders with unique order IDs and no missing values.

In [47]:
df_orders = pd.read_csv("../data/selected/orders_dataset.csv")
df_orders.info()
display(df_orders.head())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_customer_date  96476 non-null  str  
 6   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(7)
memory usage: 5.3 MB


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-16 18:17:02,2018-02-26 00:00:00


In [48]:
date_columns = [
    "order_approved_at",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for column in date_columns:
    df_orders[column] = pd.to_datetime(df_orders[column])

In [49]:
approved = df_orders["order_approved_at"]
delivered = df_orders["order_delivered_customer_date"]
estimated = df_orders["order_estimated_delivery_date"]

delivered_before_approval = delivered < approved
estimate_before_approval_day = (
    estimated.dt.normalize() < approved.dt.normalize()
)

df_orders = df_orders.loc[
    ~(delivered_before_approval | estimate_before_approval_day)
]

In [50]:
df_orders = df_orders[df_orders["order_status"] == "delivered"].dropna()
df_orders.info()

<class 'pandas.DataFrame'>
Index: 96394 entries, 0 to 99440
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96394 non-null  str           
 1   customer_id                    96394 non-null  str           
 2   order_status                   96394 non-null  str           
 3   order_purchase_timestamp       96394 non-null  str           
 4   order_approved_at              96394 non-null  datetime64[us]
 5   order_delivered_customer_date  96394 non-null  datetime64[us]
 6   order_estimated_delivery_date  96394 non-null  datetime64[us]
dtypes: datetime64[us](3), str(4)
memory usage: 5.9 MB


#### products

The selected products table has 32,951 rows, one per product_id. Two products are missing both weight and dimensions, and 623 are missing a category.

- product_id: the table key. Use it to left join products to order items with many-to-one validation.
- product_weight_g: the product weight in grams. Four products have a recorded weight of zero. Treat those weights as missing because their validity could not be verified. After replacement, six products have missing weights.
- product_length_cm, product_height_cm, and product_width_cm: dimensions in centimeters. Multiply them to estimate item volume in cubic centimeters. Keep the recorded dimensions for the products with zero recorded weight.
- product_category_name: the translated product category. Exclude it from the initial ABT, which uses weight and dimensions for product features.

Keep all product rows so that items with missing measurements still contribute to order counts, price, and freight. After joining to order items, sum item weights and volumes by order_id. Leave each corresponding order total missing if any item lacks that measurement. During model preprocessing, fill missing totals using the median calculated on the training set.

In [52]:
df_products = pd.read_csv("../data/selected/products_dataset.csv")
df_products.info()
display(df_products.head())

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   product_id             32951 non-null  str    
 1   product_weight_g       32949 non-null  float64
 2   product_length_cm      32949 non-null  float64
 3   product_height_cm      32949 non-null  float64
 4   product_width_cm       32949 non-null  float64
 5   product_category_name  32328 non-null  str    
dtypes: float64(4), str(2)
memory usage: 1.5 MB


,product_id,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name
0,1e9e8ef04dbcff4541ed26657ea517e5,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,625.0,20.0,17.0,13.0,housewares


In [62]:
df_products["product_weight_g"] = df_products["product_weight_g"].replace(
    0, float("nan")
)
print(df_products["product_weight_g"].isna().sum())

6


In [58]:
products = ["09ff539a621711667c43eba6a3bd8466", "5eb564652db742ff8f28759cd8d2652a"]
df_items[df_items["product_id"].isin(products)]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
7098,101157d4fae1c9fb74a00a5dee265c25,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-11 08:02:26,29.0,14.52
9233,1521c6bb7b1028154c8c67cf80fa809f,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-07 10:10:16,29.0,16.05
28715,415cfaaaa8cea49f934470548797fed1,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-07 10:35:19,29.0,14.52
28716,415cfaaaa8cea49f934470548797fed1,2,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-07 10:35:19,29.0,14.52
39299,595316a07cd3dea9db7adfcc7e247ae7,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-08-18 04:26:04,39.0,9.27
48424,6e150190fbe04c642a9cf0b80d83ee16,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-06-30 16:45:14,39.0,16.79
48980,6f497c40431d5fb0cfbd6c943dd29215,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-11 05:55:32,29.0,10.96
58833,85f8ad45e067abd694b627859fa57453,1,09ff539a621711667c43eba6a3bd8466,8b8cfc8305aa441e4239358c9f6f2485,2017-02-03 21:40:02,1934.0,27.00
71134,a2456e7f02197951664897a94c87242d,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-06 11:50:09,29.0,24.84
73556,a7a43f469c0d7bdb0a23a82db125aefa,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-08-28 13:15:11,39.0,15.10


#### Sellers

The sellers table has 3,095 rows and no missing values across its four columns.

- seller_id: the unique table key. Every order item has a matching seller. Join seller information to order items before aggregating by order_id.
- seller_zip_code_prefix: links to the geolocation lookup to get the origin coordinates for estimating seller-to-customer distances. Keep it as text to preserve leading zeros.
- seller_city and seller_state: candidate categorical features.

The sellers table needs no further cleaning at this stage. Seven sellers have prefixes with no match in the geolocation lookup. The left join keeps those sellers and leaves their coordinates missing.

In [11]:
df_sellers = pd.read_csv(
    "../data/selected/sellers_dataset.csv",
    dtype={"seller_zip_code_prefix": "str"}
)
df_sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   str  
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: str(4)
memory usage: 96.8 KB
